In [92]:
## Imports
import warnings
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.lines as mlines
import seaborn as sns
from scipy.stats import gaussian_kde

warnings.filterwarnings('ignore')

In [93]:
# Load data
df_pan = pd.read_csv("../datasets/raw_data/pan_raw.csv")
df_sandal = pd.read_csv("../datasets/raw_data/sandal_raw.csv")
df_sunscreen = pd.read_csv("../datasets/raw_data/sunscreen_raw.csv")
df_wallet = pd.read_csv("../datasets/raw_data/wallet_raw.csv")

In [94]:
categories = ['pan', 'sandal', 'sunscreen', 'wallet']

### Drop duplicates, fake listings, and clean data

In [95]:
for cat in ["wallet"]:
    df = globals()[f'df_{cat}']

    df = (df.sort_values('sold', ascending=False)
            .drop_duplicates(subset='id', keep='first'))

    df = df[df['rating'] > 0].copy()

    for col in ['reviews', 'sold', 'gmv_cal', 'stock']:
        cap = df[col].quantile(0.99)
        df[col] = df[col].clip(upper=cap)

    numeric_cols = ['sold', 'reviews', 'final_price', 'stock', 'favorite', 
                    'gmv_cal', 'seller_followers', 'seller_products', 'discount']
    
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna(subset=['sold', 'rating', 'reviews'])

    if 'log_rating' in df.columns:
        df = df.drop(columns=['log_rating'])

### Add z score and log

In [96]:
def zscore_series(s):
    s = pd.to_numeric(s, errors='coerce')
    std = s.std()
    if std == 0 or pd.isna(std):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.mean()) / std

In [97]:
for cat in categories:
    df = globals()[f'df_{cat}']

    df['log_reviews'] = np.log1p(df['reviews'])
    df['log_final_price'] = np.log1p(pd.to_numeric(df['final_price'], errors='coerce'))
    df['log_initial_price'] = np.log1p(pd.to_numeric(df['initial_price'], errors='coerce'))
    df['log_stock'] = np.log1p(pd.to_numeric(df['stock'], errors='coerce'))
    df['log_favorite'] = np.log1p(pd.to_numeric(df['favorite'], errors='coerce'))
    df['log_sold'] = np.log1p(df['sold'])
    df['log_gmv'] = np.log1p(pd.to_numeric(df['gmv_cal'], errors='coerce'))
    df['log_seller_followers'] = np.log1p(pd.to_numeric(df['seller_followers'], errors='coerce'))
    df['log_seller_products'] = np.log1p(pd.to_numeric(df['seller_products'], errors='coerce'))
    df['log_discount'] = np.log1p(pd.to_numeric(df['discount'], errors='coerce'))

    df['discount_percentage'] = (pd.to_numeric(df['initial_price'], errors='coerce') - 
                      pd.to_numeric(df['final_price'], errors='coerce')) / \
                     pd.to_numeric(df['initial_price'], errors='coerce') * 100
    
    df['rating_credibility'] = df['rating'] * np.tanh(df['reviews'] / 100)

    df['z_sold'] = zscore_series(df['sold'])
    df['z_rating_credibility'] = zscore_series(df['rating_credibility'])

    print(f"All columns logged, discount percentage and rating credibility created")
    print(f"Final dataset for {cat}'s shape: {df.shape[0]:,} rows × {df.shape[1]} cols")

All columns logged, discount percentage and rating credibility created
Final dataset for pan's shape: 4,551 rows × 44 cols
All columns logged, discount percentage and rating credibility created
Final dataset for sandal's shape: 10,000 rows × 47 cols
All columns logged, discount percentage and rating credibility created
Final dataset for sunscreen's shape: 3,371 rows × 46 cols
All columns logged, discount percentage and rating credibility created
Final dataset for wallet's shape: 4,151 rows × 51 cols


In [98]:
# Save dataset
for cat in categories:
    df = globals()[f'df_{cat}']
    df.to_csv(f"../datasets/final_data/{cat}_final.csv", index=False)
    print(f"Final dataset for {cat} saved with {df.shape[0]:,} rows × {df.shape[1]} cols")

Final dataset for pan saved with 4,551 rows × 44 cols
Final dataset for sandal saved with 10,000 rows × 47 cols
Final dataset for sunscreen saved with 3,371 rows × 46 cols
Final dataset for wallet saved with 4,151 rows × 51 cols
